# 6.30 - CertCF 2D Intuition Figure

Clean synthetic 2D notebook for building a CertCF atlas with `alpha = 0.3` and producing a paper-ready plot of the certified preimage approximation.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from certcf import CertCFAtlas
from certcf.eps_strategies import NearestOppositeClassClearanceStrategy
from models.classifiers import SimpleClassifier

SEED = 7
ALPHA = 0.5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASS_COLORS = {0: "#1f77b4", 1: "#d62728"}
POINT_COLORS = {0: "#2b8cbe", 1: "#e34a33"}

np.random.seed(SEED)
torch.manual_seed(SEED)

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.labelsize": 11,
    "axes.titlesize": 13,
    "legend.frameon": False,
})

In [ ]:
DATASET_CFG = {
    0: [
        {"mean": (-2.8, 1.8), "cov": ((0.55, 0.25), (0.25, 0.35))},
        {"mean": (0.0, -1.6), "cov": ((0.45, -0.18), (-0.18, 0.40))},
        {"mean": (2.8, 1.6), "cov": ((0.60, 0.22), (0.22, 0.35))},
    ],
    1: [
        {"mean": (-1.4, -0.2), "cov": ((0.50, -0.20), (-0.20, 0.45))},
        {"mean": (1.5, 0.1), "cov": ((0.50, 0.18), (0.18, 0.45))},
    ],
}


def sample_gaussian_mixture(cfg, n_per_component, rng):
    xs, ys = [], []
    for label, components in cfg.items():
        for component in components:
            xs.append(rng.multivariate_normal(component["mean"], component["cov"], n_per_component))
            ys.append(np.full(n_per_component, label, dtype=np.int64))
    X = np.vstack(xs).astype(np.float32)
    y = np.concatenate(ys)
    order = rng.permutation(len(y))
    return X[order], y[order]

rng = np.random.default_rng(SEED)
X_TRAIN, Y_TRAIN = sample_gaussian_mixture(DATASET_CFG, n_per_component=100, rng=rng)
X_TEST, Y_TEST = sample_gaussian_mixture(DATASET_CFG, n_per_component=20, rng=rng)

print(f"train: {X_TRAIN.shape}, test: {X_TEST.shape}")

In [ ]:
def train_classifier(X_train, y_train, X_test, y_test, epochs=400, batch_size=128):
    model = SimpleClassifier(input_dim=2, hidden_dims=(64, 32), num_classes=2).to(DEVICE)
    loader = DataLoader(
        TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
        batch_size=batch_size,
        shuffle=True,
    )
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    model.train()
    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            F.cross_entropy(model(xb), yb).backward()
            opt.step()

    model.eval()
    with torch.no_grad():
        train_acc = (model(torch.from_numpy(X_train).to(DEVICE)).argmax(1).cpu().numpy() == y_train).mean()
        test_acc = (model(torch.from_numpy(X_test).to(DEVICE)).argmax(1).cpu().numpy() == y_test).mean()
    print(f"train accuracy: {train_acc:.3f} | test accuracy: {test_acc:.3f}")
    return model


MODEL = train_classifier(X_TRAIN, Y_TRAIN, X_TEST, Y_TEST)

In [ ]:
def predict_np(model, X):
    model.eval()
    with torch.no_grad():
        logits = model(torch.as_tensor(X, dtype=torch.float32, device=DEVICE))
        probs = logits.softmax(dim=1).cpu().numpy()
    return probs.argmax(axis=1), probs

Y_TRAIN_MODEL, _ = predict_np(MODEL, X_TRAIN)
ATLAS_DATASET = TensorDataset(torch.from_numpy(X_TRAIN), torch.from_numpy(Y_TRAIN_MODEL.astype(np.int64)))

ATLAS = CertCFAtlas(
    MODEL,
    ATLAS_DATASET,
    DEVICE,
    cnn=False,
    norm=1,
    distance_norm=1,
    lirpa_method="backward",
    eps_strategy=NearestOppositeClassClearanceStrategy(alpha=ALPHA),
    batch_size=256,
    default_query_method="nearest_anchor",
    solver_maxiter=500,
)
ATLAS.build(build_unions=True, verbose=True)
print(ATLAS.summary())

In [ ]:
def draw_class_union(ax, atlas, label, color):
    geom = atlas.get_class_union(label)
    if geom is None or geom.is_empty:
        return
    polygons = [geom] if geom.geom_type == "Polygon" else list(getattr(geom, "geoms", []))
    for poly in polygons:
        if poly.is_empty or poly.geom_type != "Polygon":
            continue
        xs, ys = poly.exterior.xy
        ax.fill(xs, ys, color=color, alpha=0.18, linewidth=0)
        ax.plot(xs, ys, color=color, alpha=0.9, linewidth=1.15)


def plot_certcf_preimage_approximation():
    pad = 0.8
    xmin, ymin = X_TRAIN.min(axis=0) - pad
    xmax, ymax = X_TRAIN.max(axis=0) + pad
    xx, yy = np.meshgrid(np.linspace(xmin, xmax, 420), np.linspace(ymin, ymax, 420))
    grid = np.c_[xx.ravel(), yy.ravel()].astype(np.float32)
    _, probs = predict_np(MODEL, grid)
    p1 = probs[:, 1].reshape(xx.shape)

    fig, ax = plt.subplots(figsize=(7.3, 5.5))
    ax.contourf(xx, yy, p1, levels=[0.0, 0.5, 1.0], colors=[CLASS_COLORS[0], CLASS_COLORS[1]], alpha=0.07)
    ax.contour(xx, yy, p1, levels=[0.5], colors="#202124", linewidths=1.5)

    for label in (0, 1):
        draw_class_union(ax, ATLAS, label, CLASS_COLORS[label])
        mask = Y_TRAIN_MODEL == label
        ax.scatter(
            X_TRAIN[mask, 0],
            X_TRAIN[mask, 1],
            s=18,
            color=POINT_COLORS[label],
            alpha=0.72,
            edgecolor="#f8f9fa",
            linewidth=0.45,
        )

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_aspect("equal", adjustable="box")
    ax.set_axis_off()
    fig.tight_layout(pad=0)
    return fig, ax

fig, ax = plot_certcf_preimage_approximation()
plt.show()